## Welcome to this repository!

This is the description of what each script is doing in this repository and key elements to achieve the goal for each task.

Requirements: <span style="color:yellow">*pytorch*, *mujoco*, *stable_baselines3*, *gymnasium*</span>

1. <span style="color:green">*stand_walk.py*</span><br>
This script is to train humanoid to stand or walk along a certain direction under mujoco environment with reset to initial position after falling and standing position as initial position. The reward is height z with healthy height reward and penalization of action forces magnitude for standing, and additionally, x direction velocity for walking. The script uses PPO algorithm, where vf_coef is the key parameter to be set different from default.

2. <span style="color:green">*stand_walk CEM.py*</span><br>
This script is to train humanoid to continue standing or walking for extremely long period of time without falling under the same environment as script 1 (stand_walk.py). The script uses pre-trained action network from script 1, and then uses a hybrid action strategy to collect data from environment, where the hynrid action strategy is choosing to between 2 action strategy using a fixed binary distribution, and one is the pretrained action strategy trained from script 1, and another one is uniformly random action strategy. After collecting enough data from environment, a recurrent state-space model is trained as latent world model to minimize the rmse of predicted next observation and true next observation and rmse of predicted next latent state and true next latent state. One important thing is we must put enough weight to the height observation instead of other dimensions in observation space due to the scale difference, otherwise, the latent model is not guaranteed to be well trained. The script put zero weight dimensions other than x and z positions. Finally, we use the pretrained action network by sampling x trajectories of y steps ahead actions, then use the latent model to predicts the future observations recurrently given sampled actions, and then use the same reward model as environment and select K elite trajectories of actions with highest rewards. Update the action strategy distribution using the K elite trajectories of actions, repeat above for several iterations using the latest action strategy distribution. Finally, select the average of action strategy distribution from the last iteration. Tested 200,000 steps without falling, but takes long time to run in real-time.

3. <span style="color:green">*jump.py*</span><br>
This script is to train humanoid to jump with reset to initial position after falling and crouching position as initial position. The reward is height z with healthy height reward and penalization of action forces magnitude. The action space range is increased to -1 to 1 from default -0.4 to 0.4. The script uses PPO algorithm.

4. <span style="color:green">*stand_to_jump.py*</span><br>
This script is to train humanoid to jump with reset to initial position after falling and standing position as initial position. The reward is large immediate reward when height is above historical high and initial height with healthy height reward and penalization of action forces magnitude. The action space range is increased to -1 to 1 from default -0.4 to 0.4. The script uses PPO algorithm.

5. <span style="color:green">*lay_down_to_stand no_reset.py*</span><br>
This script is to train humanoid to stand without reset to initial position after falling and laying down position as initial position. The reward is a monotonically increasing function against height with penalization of action forces magnitude. The action space range is increased to -1 to 1 from default -0.4 to 0.4. The script uses SAC algorithm, where train_freq = 4096, gradient_steps = 80, batch_size = 4096. batch_size is an important parameter, if it is 256, the algorithm won't be able to learn to stand stably, but possibly stuck at seating position. And ReplayBuffer is another important thing to be changed from default. The default ReplayBuffer only keeps the latest x steps. But in this script, it is changed to keep half of the buffer to be the episodes with highest height, where each episode is 100 consecutive steps, and another half to be the latest steps. Heap structure is used to help maintain the first half of buffer for speed consideration. Another caveat, setting n_step = 1 won't work because ReplayBuffer is not complete for n_step > 1. This script is originally intended to train the humanoid to jump, but instead it stuck at standing. The script 6 successfully train the humanoid to jump.

6. <span style="color:green">*lay_down_to_jmp no_reset.py*</span><br>
This script is to train humanoid to jump without reset to initial position after falling and laying down position as initial position. The reward is the same as script 5, except an additional acceleration term which is reward for secondary derivative of height z of torso. The action space range is increased to -1 to 1 from default -0.4 to 0.4. The script uses SAC algorithm, where train_freq = 4096, gradient_steps = 80, batch_size = 6144. And ReplayBuffer is another important thing to be changed from default. In this script, it is changed to keep 1/3 of the buffer to be the episodes with highest height, where each episode is 100 consecutive steps, and another 1/3 to be the episodes with highest acceleration + a positive constant term if height is higher than healthy height for standing, where each episode is 50 consecutive steps, and another 1/3 to be the latest steps. Heap structure is used to help maintain the first 2/3 of the buffer for speed consideration. Another caveat, setting n_step = 1 won't work because ReplayBuffer is not complete for n_step > 1.